# 03 — Functional Enrichment & Subcluster Annotation

Run functional enrichment on per-subcluster DE gene lists and use the
results to assign biologically meaningful annotations.

**Workflow:**
1. Load DE results from one or more dataset runs (notebooks 01–02)
2. Filter to significantly upregulated genes per subcluster
3. Run g:Profiler enrichment via `scutils.tl.get_enriched_terms`
4. Visualise with `scutils.pl.create_pathway_dotplot`
5. Manually annotate subclusters based on enriched pathways
6. Save annotations

**Multi-dataset:** This notebook reads from multiple `RESULTS_DIRS`
entries, aggregating DE results across datasets.

## 1. Import Libraries

In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

import scutils

warnings.filterwarnings("ignore", category=FutureWarning)

## 2. Configuration

**Edit the cell below** before running the rest of the notebook.

Key parameters:
- `RESULTS_DIRS` — list of dataset output directories (from notebooks 01–02)
- `LFC_CUTOFF` / `PVAL_CUTOFF` — thresholds for selecting upregulated genes
- `ENRICHMENT_SOURCES` — g:Profiler databases to query

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║                    CONFIGURATION — edit this cell                       ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# ── Input ─────────────────────────────────────────────────────────────────────
# List of dataset result directories (each contains a de/ subfolder)
RESULTS_DIRS: list[Path] = [
    Path("results/disease_subclusters/dataset_A"),
    # Path("results/disease_subclusters/dataset_B"),
]

# ── Significance thresholds for selecting upregulated genes ───────────────────
PVAL_CUTOFF = 0.05
LFC_CUTOFF  = 0.5

# ── Enrichment settings ───────────────────────────────────────────────────────
ENRICHMENT_SOURCES: list[str] = [
    "GO:BP", "GO:CC", "GO:MF", "REAC", "WP", "KEGG",
]
ENRICHMENT_PVAL = 0.05
MAX_PATHWAYS    = 15     # max terms to show per dot-plot
ORGANISM        = "hsapiens"

# ── Pathway dot-plot colours ──────────────────────────────────────────────────
PATHWAY_COLORS = {
    "GO:BP": "#1f77b4",
    "GO:MF": "#ff7f0e",
    "GO:CC": "#2ca02c",
    "KEGG":  "#d62728",
    "REAC":  "#9467bd",
    "WP":    "#CD853F",
}

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_DIR = Path("results/disease_subclusters/functional_enrichment")
SAVE_FIGS  = True

print("Configuration loaded.")
print(f"  Result dirs    : {[str(p) for p in RESULTS_DIRS]}")
print(f"  LFC cutoff     : {LFC_CUTOFF}")
print(f"  p-value cutoff : {PVAL_CUTOFF}")
print(f"  Enrichment DB  : {ENRICHMENT_SOURCES}")
print(f"  Output dir     : {OUTPUT_DIR}")

## 3. Load DE Results & Filter Upregulated Genes

Glob `de/*.csv` from each results directory and filter to significantly
upregulated genes per subcluster.

In [ ]:
upreg_genes: dict[str, list[str]] = {}   # subcluster_label → gene list
de_tables:   dict[str, pd.DataFrame] = {} # subcluster_label → full DE table

for results_dir in RESULTS_DIRS:
    de_dir = results_dir / "de"
    csv_files = sorted(de_dir.glob("de_*.csv"))

    if not csv_files:
        print(f"⚠  No DE files found in {de_dir}")
        continue

    dataset_name = results_dir.name
    print(f"\nDataset: {dataset_name} ({len(csv_files)} files)")

    for csv_path in csv_files:
        df = pd.read_csv(csv_path)
        label = csv_path.stem.removeprefix("de_")

        # Use dataset_name prefix if aggregating multiple datasets
        key = f"{dataset_name}/{label}" if len(RESULTS_DIRS) > 1 else label

        mask = (df["padj"] < PVAL_CUTOFF) & (df["log2FoldChange"] > LFC_CUTOFF)
        genes = df.loc[mask, "gene"].dropna().unique().tolist()

        upreg_genes[key] = genes
        de_tables[key] = df
        print(f"  {key}: {len(genes)} upregulated genes")

print(f"\n✓ Loaded {len(upreg_genes)} subcluster DE results.")

## 4. Functional Enrichment per Subcluster

Run g:Profiler on each subcluster's upregulated gene list.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

enrichment_results: dict[str, pd.DataFrame] = {}

for key, genes in upreg_genes.items():
    if not genes:
        print(f"⚠  {key}: no upregulated genes — skipping enrichment.")
        continue

    print(f"\nRunning g:Profiler for {key} ({len(genes)} genes) …")

    gdf = scutils.tl.get_enriched_terms(
        genes,
        sources=ENRICHMENT_SOURCES,
        pval_adjust_sign=ENRICHMENT_PVAL,
        organism=ORGANISM,
    )

    if gdf.empty:
        print(f"  No significant terms.")
        continue

    gdf["subcluster"] = key
    enrichment_results[key] = gdf
    print(f"  {len(gdf)} significant terms.")

    # ── Dot-plot ──────────────────────────────────────────────────────────
    fig = scutils.pl.create_pathway_dotplot(
        data=gdf,
        source_colors=PATHWAY_COLORS,
        max_pathways=MAX_PATHWAYS,
        figsize=(12, 10),
        title=f"Enriched pathways — {key}",
    )

    if SAVE_FIGS:
        safe_key = key.replace("/", "_").replace("|", "_").replace(" ", "_")
        fig_path = OUTPUT_DIR / f"dotplot_{safe_key}.png"
        fig.savefig(fig_path, bbox_inches="tight")
        print(f"  Saved: {fig_path}")

    plt.show()
    plt.close(fig)

print(f"\n✓ Enrichment completed for {len(enrichment_results)}/{len(upreg_genes)} subclusters.")

## 5. Enrichment Summary Table

In [ ]:
if enrichment_results:
    all_enrichment = pd.concat(enrichment_results.values(), ignore_index=True)

    display_cols = [
        "subcluster", "source", "name", "p_value",
        "term_size", "intersection_size", "intersection_pct",
    ]
    display_cols = [c for c in display_cols if c in all_enrichment.columns]

    # Show top 5 terms per subcluster
    top_terms = all_enrichment.sort_values("p_value").groupby("subcluster").head(5)

    print(f"Top 5 enriched terms per subcluster ({len(top_terms)} rows):")
    display(
        top_terms[display_cols]
        .style
        .format({"p_value": "{:.2e}", "intersection_pct": "{:.1f}%"})
        .background_gradient(subset=["p_value"], cmap="Blues_r")
    )

    # Save full enrichment table
    enrich_path = OUTPUT_DIR / "enrichment_all_subclusters.csv"
    # Serialise list-valued columns for CSV
    save_df = all_enrichment.copy()
    if "parents" in save_df.columns:
        save_df["parents"] = save_df["parents"].apply(
            lambda p: "|".join(p) if isinstance(p, list) else p
        )
    save_df.to_csv(enrich_path, index=False)
    print(f"\nSaved: {enrich_path}")
else:
    print("No enrichment results to display.")

## 6. Annotate Subclusters

Based on the enrichment results above, assign biologically meaningful
annotations to each subcluster.

**Edit the dictionary below** with your annotations. Keys should match
the subcluster labels from the enrichment analysis.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║              SUBCLUSTER ANNOTATIONS — edit this dict                    ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# Map original subcluster label → annotation name
# Use the top enriched pathways to guide your naming.
SUBCLUSTER_ANNOTATIONS: dict[str, str] = {
    # "Microglia|AD|sub1": "DAM-like Microglia",
    # "Astrocytes|AD|sub1": "Reactive Astrocytes",
}

if SUBCLUSTER_ANNOTATIONS:
    print("Annotations:")
    for orig, annot in SUBCLUSTER_ANNOTATIONS.items():
        print(f"  {orig} → {annot}")
else:
    print("No annotations defined yet. Edit the cell above.")

In [ ]:
# Save annotations mapping
if SUBCLUSTER_ANNOTATIONS:
    annot_df = pd.DataFrame(
        list(SUBCLUSTER_ANNOTATIONS.items()),
        columns=["subcluster", "annotation"],
    )
    annot_path = OUTPUT_DIR / "subcluster_annotations.csv"
    annot_df.to_csv(annot_path, index=False)
    print(f"Saved annotations: {annot_path}")
else:
    print("No annotations to save.")

---

## Summary

| Step | Function | Key parameters |
|------|----------|----------------|
| Load DE results | `pd.read_csv()` | `RESULTS_DIRS` |
| Filter upregulated | pandas boolean mask | `PVAL_CUTOFF`, `LFC_CUTOFF` |
| g:Profiler enrichment | `scutils.tl.get_enriched_terms()` | `ENRICHMENT_SOURCES`, `ORGANISM` |
| Pathway dot-plot | `scutils.pl.create_pathway_dotplot()` | `MAX_PATHWAYS`, `PATHWAY_COLORS` |
| Annotate | Manual dict | `SUBCLUSTER_ANNOTATIONS` |

**Next:** Run `04_consensus_gene_list.ipynb` to build consensus gene
signatures across subclusters.